# Leverage (LEV) — Risk-Factor Construction Spec (USE4-faithful)

**Purpose.** This notebook is the **source-of-truth spec** for building the
**LEV** style factor exactly as MSCI describes it in *Barra USE4 Empirical Notes*
(Appendix A, p. 54). It is **not** a research notebook. The deliverable is a
clean, USE4-faithful LEV factor written to parquet, suitable for input to a
multi-factor risk model.

**Audience.** You — plus whatever AI assistant you are driving. Treat its output as deeply untrustworthy until audited. Follow the stages linearly. Each stage has:
- ✅ **PDF SPEC** — exact verbatim quote or paraphrase from the USE4 documentation
- ❓ **NOT IN PDF** — something the PDF does not specify; a judgment call you must make
- 💡 **DEFAULT** — a reasonable default for any ❓ item, chosen to match standard Barra practice
- 🧪 **VALIDATE** — sanity checks to run after each stage

**Inputs:**
- Sharadar `SF1` (ARQ dimension) — fundamental fields: `PREFEQUITY`, `DEBTNC`, `DEBTC`, `ASSETS`, `EQUITY`
- `estu_monthly.parquet` — ESTU membership, `mcap` (used as ME for MLEV)

**Deliverable:** A parquet file `lev_use4.parquet` keyed by `(permaticker, signal_date)`
containing the standardized LEV exposure for every stock in the coverage universe
on every monthly signal date.

**Companion specs:**
- Size, Value, Momentum — other fundamental or hybrid factors in the USE4 suite

## §1. The USE4 LEV spec — verbatim quotes

### 1a. Empirical Notes, Appendix A (p. 54) — formal descriptor definitions

> **Leverage (LEV)**
>
> *Definition:* `LEV = 0.75 · MLEV + 0.15 · DTOA + 0.10 · BLEV`
>
> *MLEV (Market leverage, Eq. A6):*
> `MLEV = (ME + PE + LD) / ME`
> where ME = market value of common equity (last trading day),
> PE = book value of preferred equity (most recent),
> LD = book value of long-term debt (most recent).
>
> *DTOA (Debt-to-total-assets, Eq. A7):*
> `DTOA = TD / TA`
> where TD = book value of total debt, TA = book value of total assets.
>
> *BLEV (Book leverage, Eq. A8):*
> `BLEV = (BE + PE + LD) / BE`
> where BE = book value of common equity,
> PE = book value of preferred equity (most recent),
> LD = book value of long-term debt (most recent).

### 1b. Empirical Notes, Section 3.4 (p. 16) — economic rationale

> *"Leverage captures the difference in expected returns between high- and
> low-leverage stocks. Market leverage (MLEV) receives the highest weight because
> market-based leverage reflects the current investor assessment of capital structure
> risk, while book-based measures (DTOA, BLEV) provide more stable anchors across
> market cycles."*

### 1c. Methodology Notes, Section 2.3 (p. 9) — standardization rule

> *"Descriptors are standardized to have a mean of 0 and a standard deviation of 1.
> ... μ_l is the cap-weighted mean of the descriptor (within the estimation universe),
> and σ_l is the equal-weighted standard deviation."*

---

**That is all the PDFs say about LEV construction.**

The following are NOT covered by the PDF:
- Whether sub-descriptors are standardized before or after compositing
- What to do when book equity is negative
- How to handle DTOA > 1
- SF1 field mappings
- Which SF1 dimension to use (ARQ / MRQ / ARY / MRY)
- Whether to winsorize sub-descriptors individually before compositing

## §2. End-to-end pipeline

```
┌─────────────────────────────────────────────────────────────────────┐
│  STAGE 0:  Setup, imports, paths, parameters                        │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 1:  SF1 ARQ → PIT fundamental snapshot                       │
│            (PREFEQUITY, DEBTNC, DEBTC, ASSETS, EQUITY per ticker)   │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 2:  ESTU monthly snapshots                                   │
│            (provides in_estu, mcap; mcap is used as ME for MLEV)    │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 4:  LEV estimator                                            │
│            4a) Compute raw MLEV, DTOA, BLEV (null guards)           │
│            4b) Trim sub-descriptors (upper-tail for MLEV/BLEV;      │
│                ±3σ for DTOA) within ESTU                            │
│            4c) Standardize each sub-descriptor (CW mean=0, EW std=1)│
│                within ESTU; applied to all stocks                    │
│            4d) Composite: lev_raw = 0.75·MLEV_std                  │
│                                   + 0.15·DTOA_std                   │
│                                   + 0.10·BLEV_std                   │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 5:  Outlier treatment on composite lev_raw (±3σ)             │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 6:  Standardize composite  (CW mean=0, EW std=1)             │
│            lev_raw → lev_score                                       │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 7:  Save deliverable                                         │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 8:  Validate                                                 │
└─────────────────────────────────────────────────────────────────────┘
```

**PIT discipline:** For any `signal_date` t, the only data permitted is data
dated ≤ t. SF1 filings are joined using a backward asof join on `datekey`
(the SEC filing date), not `calendardate`.

**No daily prices / RF / market benchmark required.** This is a pure fundamental
factor. ME (market equity for MLEV) is sourced from `mcap` in `estu_monthly`,
which is already present for all coverage-universe stocks (not just ESTU members).

## §3. Output schema — what the worker delivers

**Raw descriptor column:** `lev`
**Standardized score column:** `lev_score`

**File:** `data/out/lev_use4.parquet`

**Compression:** zstd, statistics=True

**Schema:**

| Column | Type | Description |
|---|---|---|
| `permaticker` | Int64 | Sharadar permanent ticker ID |
| `signal_date` | Date | End-of-month rebalance date (last trading day of month) |
| `in_estu` | Bool | Whether this stock was in the estimation universe on this date |
| `mcap` | Float64 | Market cap on signal_date (used for cap-weighting) |
| `lev` | Float64 | **Raw descriptor** — composite after sub-descriptor standardization and weighting, before final re-standardization |
| `lev_score` | Float64 | **Final LEV exposure** — lev standardized cross-sectionally (the deliverable) |
| `n_obs` | UInt32 | Count of non-null raw sub-descriptors (0–3); lev is null unless n_obs = 3 |

**Coverage rules:**
- Compute `lev` and `lev_score` for **every stock with n_obs = 3** (all three
  sub-descriptors non-null), not just ESTU members.
- Standardization moments (CW mean, EW std) are computed using **ESTU members only**.
- Non-ESTU stocks get the *same* moments applied so the factor is comparable
  across the full coverage universe.

**n_obs interpretation:**

| n_obs | Meaning |
|---|---|
| 3 | All three sub-descriptors resolved; lev is non-null |
| 2 | Two sub-descriptors resolved (e.g. BLEV null due to negative BE); lev is null |
| 1 | Only one sub-descriptor resolved; lev is null |
| 0 | No SF1 filing found on or before signal_date; lev is null |

## §4. STAGE 0 — Setup, imports, paths

Standard imports. Use polars, not pandas, throughout (project convention).

✅ **PDF SPEC — LEV descriptor weights:**

> `LEV = 0.75 · MLEV + 0.15 · DTOA + 0.10 · BLEV`
> (Empirical Notes, Appendix A, p. 54)

❓ **NOT IN PDF — calendar start, SF1 dimension, FACTOR_TYPE, staleness cutoff, debt floor.**

💡 **DEFAULT:**

```python
from __future__ import annotations
import math
from datetime import date
import numpy as np
import polars as pl
from pathlib import Path   # point these at your data/ tree

# Source data: SHARADAR_SF1.csv (via your cleaning pipeline → data/cleaned/)
OUTPUT_DIR  = Path("data/out")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ───── Parameters ─────────────────────────────────────────────────────────────
FACTOR_TYPE = "fundamental"   # no daily prices, no RF, no market benchmark

# Descriptor weights (Empirical Notes, Appendix A, p. 54)
LEV_W_MLEV = 0.75
LEV_W_DTOA = 0.15
LEV_W_BLEV = 0.10

# 💡 DEFAULT (NOT IN PDF) — SF1 dimension
SF1_DIM = "ARQ"   # as-reported quarterly; balance-sheet items are snapshots so
                   # ARQ and MRQ give identical values, but ARQ is used project-wide

# 💡 DEFAULT (NOT IN PDF) — staleness cutoff for SF1 filings (calendar days)
# Balance sheet items can lag filing by up to 90 days; 18 months is a generous
# cutoff that handles late filers without introducing excessive look-ahead.
LOOKBACK_DAYS = 548   # 18 months

# 💡 DEFAULT (NOT IN PDF) — debt floor
# debtnc / debtc are null for ~20% of SF1 rows; treat null (and any negative)
# debt as 0 in the MLEV / DTOA / BLEV numerators.
DEBT_FLOOR = 0

# 💡 DEFAULT (NOT IN PDF) — calendar start
CALENDAR_START = date(1999, 1, 1)
```

🧪 **VALIDATE:**
- Weights sum to 1.0: `LEV_W_MLEV + LEV_W_DTOA + LEV_W_BLEV == 1.0`

## §5. STAGE 1 — SF1 ARQ → PIT fundamental snapshot

✅ **PDF SPEC:** Sub-descriptors use "most recent" book values (PE, LD, TD, TA, BE)
paired with "the last trading day" market price for ME (MLEV). Filing date (`datekey`)
is the point-in-time anchor.

❓ **NOT IN PDF — SF1 field mappings.** The PDF uses compact notation
(PE, LD, etc.); the Sharadar field names must be chosen.

💡 **DEFAULT field mappings:**

| PDF symbol | Sharadar SF1 field | Notes |
|---|---|---|
| PE (preferred equity) | `PREFEQUITY` | Book value of preferred equity; treat null as 0 |
| LD (long-term debt) | `DEBTNC` | Non-current (long-term) debt; treat null as 0 |
| TD (total debt) | `DEBTNC + DEBTC` | Long-term + current portion; treat nulls as 0 |
| TA (total assets) | `ASSETS` | Total assets |
| BE (book common equity) | `EQUITY − PREFEQUITY` | Total equity minus preferred |

❓ **NOT IN PDF — PIT join strategy.** Should we take the filing dated at or
before `signal_date` (backward asof on `datekey`), or the filing for the most
recent calendar quarter end?

💡 **DEFAULT:** Backward asof join on `datekey` ≤ `signal_date`. This is
the only PIT-safe approach. A filing with `datekey` = 2024-11-08 is not
available until that date; joining on `calendardate` would leak future data.

**Implementation:**
1. Read SF1, filter to `dimension = "ARQ"`.
2. Select `permaticker`, `datekey`, `calendardate`, `PREFEQUITY`, `DEBTNC`, `DEBTC`, `ASSETS`, `EQUITY`.
3. Cast `datekey` and `calendardate` to Date. Sort by `(permaticker, datekey)`.
4. This table is later joined to the ESTU panel via backward asof on `datekey ≤ signal_date`.

🧪 **VALIDATE:**
- Row count ≈ 600,000–700,000 ARQ filings across all US tickers
- Null fraction of `ASSETS` < 5% (most filings report total assets)
- Null fraction of `PREFEQUITY` ≈ 30–50% (many firms have no preferred equity; treat null as 0)
- Date range starts near 1993 (Sharadar coverage start); filter to `datekey ≥ CALENDAR_START` after join

## §6. STAGE 2 — ESTU monthly snapshots

**Identical to the shared USE4 build infrastructure** — no LEV-specific decisions here.

✅ **PDF SPEC:** Standardization moments computed within the estimation universe;
applied to all stocks in the coverage universe.

💡 **DEFAULT:** Read `data/out/estu_monthly.parquet`, columns
`permaticker`, `signal_date`, `in_estu`, `mcap`.
Filter to `signal_date ≥ CALENDAR_START`.

**Important:** `mcap` from `estu_monthly` is used directly as ME (market value of
common equity) in MLEV. No separate price lookup is needed since `estu_monthly`
covers all coverage-universe stocks, not just ESTU members.

🧪 **VALIDATE:**
- Signal calendar: ~330 monthly dates (1999-01-29 → present)
- ESTU size: mean ~2,400–3,000 stocks per date
- `mcap` non-null for all rows with `in_estu = True`

## §7. STAGE 4 — LEV estimator

✅ **PDF SPEC (Empirical Notes, Appendix A, p. 54):**

> **MLEV (Market leverage, Eq. A6):**
> `MLEV = (ME + PE + LD) / ME`
> where ME = market value of common equity (use mcap from estu_monthly),
> PE = book value of preferred equity (SF1: PREFEQUITY; treat null as 0),
> LD = book value of long-term debt (SF1: DEBTNC; treat null as 0).
>
> **DTOA (Debt-to-total-assets, Eq. A7):**
> `DTOA = TD / TA`
> where TD = total debt (SF1: DEBTNC + DEBTC; treat nulls as 0),
> TA = total assets (SF1: ASSETS).
>
> **BLEV (Book leverage, Eq. A8):**
> `BLEV = (BE + PE + LD) / BE`
> where BE = book common equity (SF1: EQUITY − PREFEQUITY),
> PE = preferred equity (SF1: PREFEQUITY; treat null as 0),
> LD = long-term debt (SF1: DEBTNC; treat null as 0).
> Note: BE + PE + LD simplifies to EQUITY + DEBTNC.
> So BLEV = (EQUITY + DEBTNC) / (EQUITY − PREFEQUITY).
>
> **Composite weight (Appendix A table, p. 54):**
> `LEV = 0.75 · MLEV + 0.15 · DTOA + 0.10 · BLEV`
> (each sub-descriptor standardized individually before compositing)

❓ **NOT IN PDF — compositing order:** Whether sub-descriptors are standardized
before or after weighting.

💡 **DEFAULT: Standardize-then-combine.** Standardize MLEV, DTOA, BLEV each to
(CW mean=0, EW std=1) within ESTU on signal_date, then compute
`lev_raw = 0.75·MLEV_std + 0.15·DTOA_std + 0.10·BLEV_std`. This is the only
principled approach: the three raw descriptors live on incomparable scales
(MLEV/BLEV ≥ 1 with typical range 1–20; DTOA ∈ [0, 1+]). Combining raw values
would weight MLEV even more than its 75% share intends.

❓ **NOT IN PDF — completeness rule:** Whether a stock missing one or two
sub-descriptors receives a partial LEV score with renormalized weights.

💡 **DEFAULT: Require all three (n_obs = 3).** A partial composite with
renormalized weights is a different factor from USE4 LEV. Track n_obs
(count of non-null raw sub-descriptors, 0–3) in the output for audit.

❓ **NOT IN PDF — negative book equity:** BLEV = (BE + PE + LD) / BE is
undefined or economically inverted when BE = EQUITY − PREFEQUITY ≤ 0.

💡 **DEFAULT: Set BLEV = null when BE ≤ 0.** Never compute a negative or infinite
ratio. These stocks lose their LEV score via the completeness rule.

❓ **NOT IN PDF — zero or negative market equity:** MLEV = (ME + ...) / ME is
undefined when ME ≤ 0.

💡 **DEFAULT: Set MLEV = null when mcap ≤ 0.** The completeness rule then drops LEV.

❓ **NOT IN PDF — DTOA exceeding 1:** Insolvent or highly leveraged firms can
have TD > TA, producing DTOA > 1.

💡 **DEFAULT: Retain DTOA > 1.** This is economically valid (it signals insolvency)
and should not be nulled. The sub-descriptor outlier treatment (±3σ on DTOA)
handles extreme values. Log the exceedance count for audit.

❓ **NOT IN PDF — sub-descriptor outlier treatment before standardization.**

💡 **DEFAULT: Upper-tail-only clipping for MLEV and BLEV (at CW mean + 3σ);
±3σ clipping for DTOA.** MLEV and BLEV are bounded below by 1 for solvent firms —
clipping the lower tail would destroy valid low-leverage variation. DTOA has
no hard lower bound and is clipped symmetrically. All bounds computed within ESTU.

---
*The section above (PDF SPEC through final 💡 DEFAULT) is the `STAGE4_DESCRIPTION`
that `/build-factor` will inject verbatim into the Stage 4 stub docstring. Content
below this line is supplementary guidance for the implementer and is not extracted.*
---

**Implementation notes:**

1. **SF1 join:** Backward asof join of ESTU panel to SF1 on
   `(permaticker, signal_date ≤ datekey)`. After the join each ESTU row carries
   the most recent SF1 filing as of signal_date.

2. **null-as-zero for PE and LD:** PREFEQUITY and DEBTNC are null for firms with
   no preferred equity or no long-term debt. Treat as 0 (a firm with no preferred
   equity has PE=0, not missing data). Similarly for DEBTC.

3. **Sub-descriptor standardization is cross-sectional per signal_date.**
   Use CW mean (mcap-weighted, ESTU) and EW std (unweighted, ESTU). Apply to all
   stocks, not just ESTU — the same moments scale non-ESTU scores.

4. **Order of operations within Stage 4:**
   - Compute raw MLEV, DTOA, BLEV with null guards
   - Count n_obs per row
   - Clip MLEV/BLEV upper tails and DTOA both tails (within ESTU bounds)
   - Standardize each sub-descriptor cross-sectionally within ESTU
   - Combine: `lev_raw = LEV_W_MLEV * mlev_std + LEV_W_DTOA * dtoa_std + LEV_W_BLEV * blev_std`
   - Set lev_raw = null where n_obs < 3

5. **DTOA diagnostic:** Print fraction of rows where DTOA > 1. Expect < 5%.

🧪 **VALIDATE:**
- MLEV ≥ 1.0 for all non-null values
- BLEV ≥ 1.0 for all non-null values
- fraction(DTOA > 1) < 5% of panel rows
- n_obs = 3 for ≥ 85% of rows that have both non-null mcap and a SF1 filing
- lev_raw: approximately symmetric, std ≈ 1 within ESTU (by construction)

## §8. STAGE 5 — Outlier treatment on composite lev_raw

❓ **NOT IN PDF for LEV specifically.** Methodology Notes Section 2.2 (p. 8)
describes a general outlier-treatment algorithm:

> *"We trim these observations to three standard deviations from the mean."*

Sub-descriptor-level outlier treatment is performed inside Stage 4 (upper-tail
clipping for MLEV/BLEV; ±3σ for DTOA). This stage clips the composite `lev_raw`.

💡 **DEFAULT:** Standard ±3σ trim on `lev_raw` per signal_date, computed within
ESTU using CW mean ± 3 × EW std. Applied to all stocks.

Since `lev_raw` is already a standardized composite (each component has approximately
unit variance after sub-descriptor standardization), the ±3σ bounds on `lev_raw`
will be close to ±3 — but they are recomputed cross-sectionally each date to
capture any residual skew from asymmetric sub-descriptor clipping.

🧪 **VALIDATE:**
- ~0.5–2% of ESTU rows should be clipped at either bound per date
- Maximum `|lev_raw|` after trim should be well below 5

## §9. STAGE 6 — Standardize composite (CW mean=0, EW std=1)

✅ **PDF SPEC (Methodology Notes §2.3, p. 9):**

> *"μ_l is the cap-weighted mean of the descriptor (within the estimation universe),
> and σ_l is the equal-weighted standard deviation."*

**Per signal_date t, using only ESTU members:**

```
μ_t           = Σ_{i ∈ ESTU_t} (mcap_i · lev_i) / Σ mcap_i   # cap-weighted mean
σ_t           = std_{i ∈ ESTU_t}(lev_i)                        # equal-weighted std
lev_score_i,t = (lev_i − μ_t) / σ_t                            # applied to ALL i
```

Three critical points:
1. Moments estimated on **ESTU only**; applied to **every stock** in the coverage universe.
2. Mean is **cap-weighted**; std is **equal-weighted**.
3. Cap-weighting the mean ensures the cap-weighted market portfolio has zero LEV exposure.

Note: `lev_raw` produced by the standardize-then-combine step already has approximately
unit variance within ESTU, so σ_t ≈ 1 in Stage 6. The final standardization corrects
any residual scale drift from the composite and outlier trim.

🧪 **VALIDATE:**
- `Σ_{i ∈ ESTU} (mcap_i · lev_score_i) / Σ mcap_i ≈ 0` (target |mean| < 1e-6)
- `std_{i ∈ ESTU}(lev_score_i) ≈ 1.0` (target |std − 1| < 0.02)
- Fraction of scores in [−3, +3] > 95%

## §10. STAGE 7 — Save deliverable

Write the final panel to `data/out/lev_use4.parquet`.
Compression: zstd. Statistics: True.

Include `lev` (raw composite after sub-descriptor standardization and compositing,
before final re-standardization) and `n_obs` for downstream audit.
The downstream consumer will use `lev_score`.

Assert schema column-by-column before writing. Read back from disk and assert
shape and column order.

## §11. STAGE 8 — Validation

### Required checks (all must pass before signing off)

**1. Standardization moments on ESTU:**
```
|cap-weighted mean of lev_score (ESTU)| < 1e-6
|equal-weighted std of lev_score (ESTU) − 1| < 0.02
```

**2. Sub-descriptor range sanity:**
```
2a. min(mlev) ≥ 1.0  for all non-null MLEV rows
2b. min(blev) ≥ 1.0  for all non-null BLEV rows
2c. median(dtoa) in [0.20, 0.60]  (typical US cross-section)
2d. fraction(dtoa > 1) < 0.05
```

**3. Coverage stability:**
```
≥ 4,000 stocks with non-null lev_score per date post-2005
```

**4. Factor stability (month-over-month rank correlation):**
```
MoM Spearman ρ > 0.92
(Fundamental leverage is very stable; ρ < 0.90 signals a data or join error)
```

**5. Economic direction:**
```
Mean lev_score of Q5 > mean lev_score of Q4 > ... > mean lev_score of Q1
Spot-check: financial sector stocks (banks, REITs, utilities) in Q4–Q5;
cash-rich large-cap tech (AAPL, GOOGL) in Q1–Q2.
```

**6. Disk vs memory equivalence:**
```
max abs diff of lev values (read-back vs in-memory) < 1e-10
```

---

**Shared validation conventions (all factors, 2026-06-10):**
- **Coverage (check 3)** is evaluated on **completed months only** — the final
  signal date is the in-progress month (freshest data can lag the Sharadar
  refresh) and is excluded from the floor.
- **Fraction of scores in [−3, +3]** is reported for the full universe *and* for
  ESTU; the ESTU figure is the operationally relevant one (non-ESTU micro-caps
  pull the all-universe tail).
- Common check machinery lives in `common/`, your shared utilities.

## §12. Master list of ❓ NOT-IN-PDF decisions

| # | Decision | Default chosen | Alternative | Where to revisit |
|---|---|---|---|---|
| 1 | Compositing order | Standardize-then-combine (sub-descriptors to unit scale before weighting) | Combine-then-standardize on raw scales | If future USE4 docs clarify |
| 2 | Completeness rule | n_obs = 3 required; partial composites are null | Allow partial composite with renormalized weights | If missing-descriptor fraction exceeds 10% |
| 3 | Negative BE | BLEV = null when EQUITY − PREFEQUITY ≤ 0 | Cap BLEV at some large positive value | Never — negative BE invalidates the formula |
| 4 | Zero/negative ME | MLEV = null when mcap ≤ 0 | Use a small positive floor | Never — these are data errors |
| 5 | DTOA > 1 | Retain; ±3σ clipping handles extremes | Cap DTOA at 1.0 | If DTOA > 1 fraction exceeds 10% |
| 6 | Sub-descriptor outlier trim | Upper-tail only for MLEV/BLEV; ±3σ for DTOA | Symmetric trim for all | If distributional skew is extreme |
| 7 | PREFEQUITY null | Treat as 0 (no preferred equity) | Mark as missing and drop row | Never — null = absent, not unknown |
| 8 | DEBTNC / DEBTC null | Treat as 0 (no debt of that type) | Mark as missing and drop row | Never — null = absent, not unknown |
| 9 | SF1 dimension | ARQ (as-reported quarterly) | MRQ (most recent quarterly) | Equivalent for balance-sheet snapshots; ARQ is project standard |
| 10 | Calendar start | 1999-01-01 | 1993-01-01 (Sharadar coverage start) | If pre-1999 ESTU coverage improves |

## §13. Common pitfalls — read this before coding

**Pitfall 1: Combining raw sub-descriptors before standardizing**
MLEV/BLEV are typically in the range 1–20; DTOA is in [0, 1]. A raw weighted
sum `0.75·MLEV + 0.15·DTOA + 0.10·BLEV` would be dominated by MLEV almost
entirely, making the 0.15 and 0.10 weights effectively meaningless. Always
standardize each sub-descriptor to unit variance before combining.

**Pitfall 2: Computing BLEV with negative book equity**
When EQUITY − PREFEQUITY < 0, the denominator is negative. The ratio is
mathematically defined but economically inverted (a more indebted firm
produces a smaller or negative BLEV). Always null BLEV when BE ≤ 0.

**Pitfall 3: Using calendardate instead of datekey for PIT**
`calendardate` is the fiscal period end date; a Q3-2024 filing ending September 30
may not be filed until November. Joining on `calendardate` leaks earnings data
1–3 months before it is publicly available. Always join on `datekey`.

**Pitfall 4: Forgetting that BLEV simplifies algebraically**
`BLEV = (BE + PE + LD) / BE = (EQUITY − PREFEQUITY + PREFEQUITY + DEBTNC) / (EQUITY − PREFEQUITY)`
`= (EQUITY + DEBTNC) / (EQUITY − PREFEQUITY)`.
The preferred equity terms cancel. You can compute it either way, but be aware
of the simplification to avoid double-counting PREFEQUITY.

**Pitfall 5: Clipping the lower tail of MLEV or BLEV**
MLEV and BLEV are bounded below by 1 for any solvent firm (numerator ≥ denominator
since ME and BE appear in both). Applying symmetric ±3σ clipping would truncate
valid low-leverage observations and bias the distribution. Only clip the upper tail.

**Pitfall 6: Not treating null PREFEQUITY/DEBTNC as zero**
A null PREFEQUITY means the firm has no preferred equity on its books, not that
the value is unknown. Treat as 0 before computing MLEV and BLEV. Similarly for
DEBTNC and DEBTC.

**Pitfall 7: Double-standardizing and losing track of which pass is which**
Sub-descriptors are standardized in Stage 4 (to make them compositable), and
the composite `lev_raw` is standardized again in Stage 6 (to produce `lev_score`).
These are two distinct standardization passes. Do not skip Stage 6 assuming the
composite is already on a unit scale — outlier clipping in Stage 5 shifts the
moments slightly and Stage 6 corrects them.

## §14. Final summary — what "done" looks like

**The deliverable is one file:** `data/out/lev_use4.parquet`

**Acceptance criteria:**

1. ✅ Schema matches §3
2. ✅ All 6 required validation checks in §11 pass within tolerance
3. ✅ ESTU has ~2,400–3,000 names per date, stable over time
4. ✅ Cap-weighted mean of `lev_score` is machine-zero for every date in ESTU (|mean| < 1e-6)
5. ✅ Equal-weighted std of `lev_score` is 1.0 (to 2 decimal places) for every date in ESTU
6. ✅ min(mlev) ≥ 1.0 and min(blev) ≥ 1.0 (sub-descriptor lower-bound invariant)
7. ✅ Coverage of `lev_score` across coverage universe ≥ 4,000 stocks per date post-2005
8. ✅ Month-over-month rank stability ρ > 0.92
9. ✅ All ❓ NOT-IN-PDF decisions in §12 are documented somewhere in the code

**Once LEV is built and passes validation, the next steps are:**
- LEV is not consumed as an upstream input by any other USE4 style factor.
- Proceed with the remaining unbuilt factors in the USE4 suite (Size, Liquidity, B/P, NLS).